# Xarray-Spatial Reproject: bounds policy at projection singularities

Reprojecting a raster near the antimeridian or a pole can produce huge or undefined output extents. xrspatial.reproject ships heuristics that crop the output to a sane size, but those heuristics fire silently and can discard real data. The `bounds_policy` parameter lets you pick the trade-off explicitly.

### What you'll build

1. Project a global geographic raster to Web Mercator under each of the four policies.
2. Compare the resulting output bounds side by side.
3. Surface the `UserWarning` emitted under the default `auto` policy.
4. Walk through the geographic clamp on a near-antimeridian raster.

![preview](images/55_reproject_bounds_policy_preview.png)

- [Setup](#Setup)
- [The four policies](#The-four-policies)
- [Auto policy emits a warning](#Auto-policy-emits-a-warning)
- [Geographic clamp on a near-antimeridian raster](#Geographic-clamp-on-a-near-antimeridian-raster)
- [References](#References)

Standard imports. The reproject function is the only thing we need from xrspatial.

In [ ]:
import warnings

import numpy as np
import xarray as xr

import matplotlib.pyplot as plt

from xrspatial.reproject import reproject

## Setup

Build a near-global geographic raster (EPSG:4326). The longitude range stops just short of +/-180 so we still have a meaningful projected extent, and the latitudes climb to +/-85 to put us near the Web Mercator polar singularity.

In [ ]:
np.random.seed(42)
lats = np.linspace(85, -85, 80)
lons = np.linspace(-178, 178, 160)
yy, xx = np.meshgrid(lats, lons, indexing='ij')
data = (np.sin(np.deg2rad(xx) * 2) * np.cos(np.deg2rad(yy)) + 1.0) / 2.0

raster = xr.DataArray(
    data.astype(np.float32),
    dims=['y', 'x'],
    coords={'y': lats, 'x': lons},
    attrs={'crs': 'EPSG:4326'},
)
raster

The source values are smooth east-west bands modulated by latitude. The exact pattern is incidental. What matters is the spatial extent.

In [ ]:
raster.plot.imshow(size=4, aspect=2, cmap='viridis', add_colorbar=False)
plt.title('Source raster (EPSG:4326)')
plt.show()

## The four policies

Run the same reprojection four times, once per policy. Suppress the warnings for now and just compare the resulting output bounds and shapes.

In [ ]:
policies = ['auto', 'raw', 'clamp', 'percentile']
results = {}
for policy in policies:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', UserWarning)
        out = reproject(raster, 'EPSG:3857', resolution=2e5, bounds_policy=policy)
    results[policy] = out

import pandas as pd
rows = []
for policy, out in results.items():
    x = out.coords['x'].values
    y = out.coords['y'].values
    rows.append({
        'policy': policy,
        'shape': out.shape,
        'x_min': float(x.min()),
        'x_max': float(x.max()),
        'y_min': float(y.min()),
        'y_max': float(y.max()),
    })
pd.DataFrame(rows)

`auto` and `percentile` produce the smallest extents because the 2/98 percentile fallback throws away the extremes near the poles. `raw` keeps every projected pixel and ends up the largest. `clamp` sits in between because it trims the source extent inward by 0.01 deg before projecting, but it still keeps all the projected output.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, policy in zip(axes.flat, policies):
    out = results[policy]
    out.plot.imshow(ax=ax, cmap='viridis', add_colorbar=False)
    ax.set_title(f"bounds_policy={policy!r}  shape={out.shape}")
plt.show()

<div class="alert alert-block alert-warning">
<b>Picking a policy.</b> Use <code>raw</code> when you need the true projected extent of your input footprint, for example when mosaicking many tiles or computing area statistics. Use <code>auto</code> (or <code>percentile</code>) when you'd rather lose a few pixels near a singularity than allocate a giant output grid.
</div>

## Auto policy emits a warning

Under `auto`, when the percentile fallback or the geographic clamp actually alters the bounds, reproject emits a `UserWarning`. The warning names the policy and reports the per-side delta vs the raw projected bounds.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    reproject(raster, 'EPSG:3857', resolution=2e5, bounds_policy='auto')

for w in caught:
    if issubclass(w.category, UserWarning) and 'bounds_policy' in str(w.message):
        print(w.message)
        print('---')

If you intentionally want the silent cropping back, suppress the warning:

In [ ]:
with warnings.catch_warnings():
    warnings.filterwarnings(
        'ignore', category=UserWarning, message='.*bounds_policy.*'
    )
    out = reproject(raster, 'EPSG:3857', resolution=2e5)
    print(f'output shape: {out.shape}')

## Geographic clamp on a near-antimeridian raster

When the source CRS is geographic and the source extent touches +/-180 longitude, the default policy trims it inward by 0.01 deg before projecting. This avoids infinities at the antimeridian, but it also trims a sliver of real data. Surface the trim with the `clamp` policy on a synthetic raster whose extent runs to exactly +/-180 longitude.

In [ ]:
edge_raster = xr.DataArray(
    np.random.RandomState(0).rand(40, 80).astype(np.float32),
    dims=['y', 'x'],
    coords={'y': np.linspace(60, -60, 40),
            'x': np.linspace(-180, 180, 80)},
    attrs={'crs': 'EPSG:4326'},
)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    out_clamp = reproject(
        edge_raster, 'EPSG:3857', resolution=2e5, bounds_policy='clamp'
    )

for w in caught:
    if issubclass(w.category, UserWarning) and 'clamp' in str(w.message):
        print(w.message)
        print('---')

The warning names the original source extent and the trimmed one. If you need the antimeridian-touching data preserved, switch to `bounds_policy='raw'` and handle any projection infinities downstream.

<div class="alert alert-block alert-info">
<b>Antimeridian wrap.</b> Geographic rasters that genuinely span the antimeridian are still tricky. Even with <code>raw</code>, single-pass reprojection cannot reconstruct a discontinuous footprint. Consider splitting the raster on the antimeridian and reprojecting each half separately, then merging.
</div>

## References

- xarray-spatial reproject reference: https://xarray-spatial.org/reference/reproject.html
- Web Mercator (EPSG:3857) polar truncation: https://epsg.io/3857
- Issue #2187: bounds heuristics audit